# Part 4 · Classical CSP of T2 with mol-CSPy  (~8 min)

[mol-CSPy](https://gitlab.com/mol-cspy/mol-cspy) (Day group, Southampton) is a full CSP package. Its generator fills
chosen space groups with trial structures from a Sobol sequence; in production each one is then minimised with DMACRYS.
We run the generator here and relax with MACE-OFF, as in Part 3, so the two searches can be compared.

In [ ]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit average-minimum-distance toml "requests==2.32.4"
import csp_utils as cu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

## 4.1 Get mol-CSPy (~10 s)

In [ ]:
cspy = cu.get_molcspy()     # Python sources only, nothing to compile
print("mol-CSPy", cspy.__version__)
HAVE_RUN = (cu.T2_MOLCSPY / "endpoints.csv").exists()   # 100 structures relaxed before the school

## 4.2 Sobol vs random numbers

Each trial structure is set by up to 12 numbers (cell, position, orientation). Two of them, 256 points:

In [ ]:
n = 256
pseudo = np.random.default_rng(0).random((n, 2))
sobol = np.array([cu._sobol_vector(i, 2) for i in range(1, n + 1)])
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
for ax, pts, t in [(axs[0], pseudo, "random"), (axs[1], sobol, "Sobol")]:
    ax.scatter(*pts.T, s=8, c="#893F0D"); ax.set_title(t); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

> **Question 11.** Which one leaves gaps? Why does that matter when you can only afford a few thousand trials per space group?

## 4.3 Generate T2 crystals

In [ ]:
import time
gen = {}
for sg in (2, 14):           # P-1 and P2_1/c
    t0 = time.time()
    atoms, seeds, tried = cu.molcspy_generate(sg, 50)
    gen[sg] = (atoms, seeds)
    print(f"space group {sg}: {len(atoms)} structures from {tried} Sobol points in {time.time() - t0:.1f} s")
plt.figure(figsize=(7, 3))
for sg, c in [(2, "#893F0D"), (14, "#C9A27E")]:
    plt.hist([cu.density(a) for a in gen[sg][0]], bins=20, range=(0.1, 1.8), alpha=.7, color=c, label=f"space group {sg}")
for r in cu.T2_REFS:
    plt.axvline(cu.t2_reference(r).density, c="k", ls=":", lw=1)
plt.xlabel("density before relaxation (g/cm$^3$)"); plt.legend(fontsize=8);

In [ ]:
cu.show(gen[2][0][0], supercell=(2, 2, 2))

## 4.4 Relax one (~45 s)

The same Sobol point gives the same structure everywhere, so we can look up how this one ended in the full run.

In [ ]:
k = 0                        # any index below 50
at, seed = gen[2][0][k], gen[2][1][k]
name = f"T2_sg2_s{seed:06d}"
t0 = time.time()
r = cu.relax(at, fmax=0.1, steps=60, symmetry=False)
print(f"{name}: {time.time() - t0:.0f} s, density {cu.density(at):.3f} -> {cu.density(r):.3f} g/cm3, "
      f"E = {cu.energy_per_molecule(r, n_mol=len(r) // 46) - cu.T2_E0_KJMOL:.0f} kJ/mol")
if HAVE_RUN:
    M = cu.molcspy_endpoints()
    display(M[M.name == name][["name", "converged", "steps", "density", "rel_energy_kjmol"]])

> **Question 12.** A mol-CSPy structure is nowhere near a real crystal until it has been relaxed. Compare with 3.2:
> what does that mean for the cost of a classical search?

## 4.5 100 mol-CSPy structures vs 100 Clari-H samples

In [ ]:
C = cu.t2_endpoints(); C = C[C.converged]
if HAVE_RUN:
    M = cu.molcspy_endpoints(); M = M[M.converged & M.intact]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(C.density, C.rel_energy_kjmol, s=14, c="#BDBDBD", label="Clari-H")
    ax.scatter(M.density, M.rel_energy_kjmol, s=14, c="#893F0D", alpha=.7, label="mol-CSPy")
    ax.set_ylim(-5, 270); ax.set_xlabel("density (g/cm$^3$)"); ax.set_ylabel("relative MACE-OFF energy (kJ/mol per T2)")
    ax.legend(fontsize=8)
    CS = cu.molcspy_clari_scores()
    table = {who: {**{"relaxed and converged": len(d)},
                   **{f"{r}: 15/15": int((s[f"n_matched_{r}"] == 15).sum()) for r in cu.T2_REFS},
                   "lowest E (kJ/mol)": f"{d.rel_energy_kjmol.min():.1f}"}
             for who, d, s in [("Clari-H", C, CS), ("mol-CSPy", M, M)]}
    display(pd.DataFrame(table, dtype=object))
else:
    print("data/t2_molcspy/ is missing")

> **Question 13.** With one molecule in the asymmetric unit, our space groups give 2, 4 or 8 molecules per cell.
> γ and the global minimum have 6. Which references could this mol-CSPy run find at all?
>
> **Question 14.** Give one advantage of each method. What would you change in mol-CSPy to find γ?

---
mol-CSPy: gitlab.com/mol-cspy/mol-cspy · Case *et al.*, *J. Chem. Theory Comput.* **12**, 910 (2016) ·
Taylor, Butler & Day, *Faraday Discuss.* **256**, 434 (2025)